In [4]:
# Install libraries

import pyodbc
import pandas as pd
import warnings
import numpy as np
from scipy import stats 
from scipy.stats import chi2_contingency
warnings.filterwarnings("ignore")

In [5]:
# Set Connection

conn = pyodbc.connect(
    "Driver={SQL Server};"
    "Server=NETHRA\SQLEXPRESS;"
    "Database=BankChurn;"
    "Trusted_Connection=yes;"
)

cursor = conn.cursor()

## Test 1: Independent Samples t-Test

### Business Question

In the banking industry, account balance is one of the most critical indicators of customer value. The business wants to understand:

> **Do customers who leave the bank (churn) differ in account balance compared to loyal customers?**

The answer to this question can influence several strategic decisions, including:

- Retention campaign targeting
- VIP protection programs
- Risk segmentation
- Revenue forecasting

**Business Interpretation:**

- **If high-balance customers are more likely to churn:** There may be a significant risk of losing high-value clients, which could justify targeted retention and protection strategies.
- **If low-balance customers churn more frequently:** This may indicate issues related to customer engagement, product fit, or the bank's value proposition.

---

### Statistical Hypotheses

We aim to compare the **mean account balance** between two independent customer groups:

- Customers who have **churned**
- Customers who have been **retained (loyal)**

**Null Hypothesis ($H_0$):**

> The mean account balance is equal between churned and retained customers.

$$
H_0: \mu_{churned} = \mu_{retained}
$$

**Alternative Hypothesis ($H_1$):**

> The mean account balance differs between churned and retained customers.

$$
H_1: \mu_{churned} \neq \mu_{retained}
$$

This is a **two-tailed hypothesis test** because the objective is to determine whether a difference exists, regardless of which group has the higher mean balance.

---

### Column Selection

Only two columns are required for this hypothesis test:

| Column | Type | Purpose |
|---|---|---|
| `Balance` | Continuous Numerical | Measure whose mean is being compared |
| `Churned` | Binary Categorical | Defines churned and retained customer groups |

---

### Test Assumptions

The Independent Samples t-Test relies on the following assumptions:

**1. Independence**

Observations should be independent of one another. Each customer should represent a unique observation in the dataset.

**2. Normality**

`Balance` should be approximately normally distributed within each customer group.

For sufficiently large samples, the t-test is generally robust to moderate deviations from normality because of the Central Limit Theorem.

**3. Homogeneity of Variance**

The variance of `Balance` should be approximately equal between churned and retained customers.

This assumption can be evaluated using **Levene's Test**.

- If `p-value > 0.05` → there is insufficient evidence that the variances differ, so the equal-variance assumption is reasonable.
- If `p-value ≤ 0.05` → there is evidence that the variances differ, so **Welch's t-test** should be preferred.

---

### Why This Test?

The Independent Samples t-Test is appropriate because:

- **Dependent variable:** `Balance` — Continuous numerical variable
- **Independent variable:** `Churned` — Binary categorical variable
- **Number of groups:** Two independent groups
- **Objective:** Compare the mean account balance between churned and retained customers

Therefore, an **Independent Samples t-Test** (or Welch's t-test when equal variances are not supported) is appropriate for evaluating whether the two groups differ in mean account balance.

##### Implement Test

 Step 1 — Data Preparation

In [6]:
query = """
SELECT
    a.Balance,
    d.Churned
FROM account a
JOIN demographic d
    ON a.CustomerId = d.CustomerId
"""

df_t_test = pd.read_sql(query, conn)
df_t_test.head()

,Balance,Churned
0,119827.49,True
1,83807.86,False
2,159660.80,True
3,119827.49,False
4,125510.82,False



 Step 2 — Split Groups

In [7]:

churned_t_test = df_t_test[df_t_test["Churned"] == 1]["Balance"]
retained_t_test = df_t_test[df_t_test["Churned"] == 0]["Balance"]

Step 3 — Descriptive Stats

In [8]:
print("Churned Mean:", churned_t_test.mean())
print("Retained Mean:", retained_t_test.mean())

Churned Mean: 120521.27620520373
Retained Mean: 119650.01690066556


Step 4 — Variance Check

In [9]:
# if p > 0.05, equal variance assumed
stats.levene(churned_t_test, retained_t_test)

LeveneResult(statistic=np.float64(36.69675448689501), pvalue=np.float64(1.4298234841363054e-09))

Step 5 — Run t-test


### **Variance Assumption Conclusion**

Since Levene's Test indicates that the variances between the churned and retained customer groups are **not equal**, the equal variance assumption is violated.

Therefore, we should use **Welch's t-test** instead of the standard Independent Samples t-Test.

Welch's t-test is appropriate because it does not assume equal variances between the two groups.

In [10]:
t_stat, p_value = stats.ttest_ind(
    churned_t_test,
    retained_t_test,
    equal_var = False 
)

print("t-stat:", t_stat)
print("p-value:", p_value)

t-stat: 1.3534724523393942
p-value: 0.17601045694361078


#### **Output Interpretation**

The p-value is greater than the significance level of **0.05**. Therefore, we **fail to reject the null hypothesis ($H_0$)**.

This means that there is **insufficient statistical evidence to conclude that the mean account balance differs significantly between churned and retained customers**.

Therefore, based on this test, account balance does not show a statistically significant association with churn through a difference in group means.

> **Business Conclusion:** The average account balance of churned customers is not statistically significantly different from that of retained customers. Therefore, `Balance` alone may not be a strong indicator for distinguishing churned customers from retained customers in this dataset.

**Note:** This test identifies differences in mean balance between the two groups; it does not establish whether account balance causes or does not cause customer churn.

## **Test 2: Chi-Square Test of Independence**

### **Business Question**

In the banking industry, customer activity and engagement can be important indicators of churn risk. The business wants to understand:

> **Are inactive customers more likely to churn?**

Understanding whether customer activity is associated with churn can support several strategic initiatives:

- Early Warning System development
- Reactivation campaign design
- Customer Health Score modeling
- Behavior-based retention strategies

If customer activity is significantly associated with churn, activity status may be useful when identifying customers who require additional engagement or retention efforts.

### **Statistical Hypotheses**

We are examining the relationship between two categorical variables:

- `Churned` → Churned / Retained
- `IsActive` → Active / Inactive

**Null Hypothesis ($H_0$):**

> Customer churn and customer activity status are independent.

$$
H_0: \text{Churned and IsActive are independent}
$$

**Alternative Hypothesis ($H_1$):**

> Customer churn and customer activity status are associated.

$$
H_1: \text{Churned and IsActive are associated}
$$

The significance level used for this test is:

$$
\alpha = 0.05
$$


### **Column Selection**

Two categorical variables are required for this hypothesis test:

| Column | Type | Description |
|---|---|---|
| `IsActive` | Binary Categorical | Indicates whether the customer is active or inactive |
| `Churned` | Binary Categorical | Indicates whether the customer churned or was retained |

The objective is to determine whether there is a statistically significant association between customer activity and churn.

##### *Implement Test*

Step 1 — Data Preparation

In [13]:
query = """
SELECT
    d.Churned,
    a.IsActive
FROM demographic d
JOIN account a
    ON d.CustomerId = a.CustomerId
"""

df_chi = pd.read_sql(query, conn)
df_chi.head()


,Churned,IsActive
0,True,True
1,False,True
2,True,False
3,False,False
4,False,True


Step 2 — Build Contingency Table

In [12]:

ct = pd.crosstab(df_chi["Churned"], df_chi["IsActive"])
print(ct)

IsActive  False  True 
Churned               
False      3547   4416
True       1302    735


Step 3 — Run Chi-Square Test

In [14]:
chi2, p_value, dof, expected = chi2_contingency(ct)

print("Chi2:", chi2)
print("p-value:", p_value)
print("Degrees of Freedom:", dof)

Chi2: 242.98534164287963
p-value: 8.785858269303703e-55
Degrees of Freedom: 1


Step 4 — Expected Frequencies for checking assumption

In [15]:

print(expected)

[[3861.2587 4101.7413]
 [ 987.7413 1049.2587]]


### Output Interpretation

### **Output Interpretation**

The p-value is **less than 0.05**, which is below the significance level ($\alpha = 0.05$). Therefore, we **reject the null hypothesis ($H_0$)**.

This provides statistically significant evidence that **customer activity (`IsActive`) and customer churn (`Churned`) are associated (dependent)**.

> **Business Conclusion:** Customer activity status is significantly associated with customer churn. This suggests that activity level may be useful for identifying customers with different churn behavior and can be considered when developing customer engagement and retention strategies.

**Note:** The Chi-Square test establishes an **association**, not causation. Therefore, we cannot conclude from this test alone that customer inactivity causes churn.

## **Test 3: One-Way ANOVA**

### **Business Question**

The bank operates across multiple countries/regions. The business wants to understand:

> **Does customer tenure differ significantly across countries?**

### **Why Is This Question Important?**

If customer tenure differs across countries, it may indicate differences in customer relationship duration across markets.

Lower tenure in certain countries may be associated with factors such as:

- Customers maintaining shorter relationships with the bank
- Differences in service quality or customer experience
- Competitive market conditions
- Differences in product offerings
- Differences in digital banking experience
- Different customer acquisition and attrition patterns

The results can support strategic decisions such as:

- Designing country-specific retention strategies
- Investigating markets with relatively low customer tenure
- Improving products, services, or channels in specific markets
- Benchmarking customer relationship patterns across countries

#### **Statistical Hypotheses**

- **Null Hypothesis:** The average tenure is the same across all countries.
- **Alternative Hypothesis:** The average tenure is not the same across all countries.

#### **Column Selection**

To perform this hypothesis test, only two columns are required:

- `Tenure` is the **numerical variable** that represents how long a customer has been using the bank's services.
- `Geography` is the **categorical variable** that represents the different countries/locations of the customers.

#### **Test Assumptions**

To ensure the validity of the ANOVA test, the following assumptions should be evaluated:

- **Independence of Observations**: Each customer should represent an independent observation. In this dataset, each customer is recorded once, so the observations are considered independent.

- **Normality Within Each Group**: The distribution of `Tenure` within each country should be approximately normal. However, ANOVA is relatively robust to moderate violations of normality, particularly when each group has a sufficiently large sample size.

- **Homogeneity of Variances**: The variance of `Tenure` should be reasonably similar across countries. This assumption can be assessed using **Levene's Test**.
  - If the Levene's Test p-value is **greater than 0.05**, we fail to reject the null hypothesis and can proceed with the standard **One-Way ANOVA**.
  - If the Levene's Test p-value is **less than or equal to 0.05**, the equal variance assumption is violated, and **Welch's ANOVA** should be considered instead.

- **Adequate Sample Size per Group**: Each country should have a sufficient number of observations to provide reliable estimates of the group means and variances.

#### **Why this test?**

The choice of statistical test is based on the structure of the variables and the analytical objective:

- Dependent variable is Numerical (`Tenure`)
- Independent variable is Categorical with multiple groups (`Geography`)
- Objective is to compare the average `Tenure` across multiple countries

Therefore, **One-Way ANOVA** is appropriate because it is used to compare the mean of a numerical variable across three or more independent groups.

#### Implement Test

Step 1 — Data Preparation

In [16]:
query = """
SELECT
    l.Geography AS Country,
    a.Tenure
FROM account a
JOIN demographic d
    ON a.CustomerId = d.CustomerId
JOIN location l
    ON d.LocationId = l.LocationId
WHERE a.Tenure IS NOT NULL
  AND l.Geography IS NOT NULL;
"""

df_anova = pd.read_sql(query, conn)
df_anova.head()

,Country,Tenure
0,Spain,9
1,Germany,1
2,USA,1
3,Spain,6
4,Spain,10


Step 2 — Descriptive stats

In [17]:
summary = df_anova.groupby("Country")["Tenure"].agg(["count", "mean", "std"])
print(summary.sort_values("mean", ascending=False))

         count      mean       std
Country                           
UK        1703  5.127422  2.939195
France    1714  5.042590  2.904582
USA       1669  5.002397  2.890694
Germany   1626  4.987700  2.870085
Canada    1679  4.982132  2.888003
Spain     1609  4.927906  2.857046


Step 3 — Check variance equality

In [18]:
# if p > 0.05, equal variance assumed

groups = [g["Tenure"].values for _, g in df_anova.groupby("Country")]
lev = stats.levene(*groups)
print(lev)

LeveneResult(statistic=np.float64(0.5338267546333629), pvalue=np.float64(0.7508283480014718))


Since, p-value is more than 0.05, H0 is rejected. So variances are not equal.

 Step 4 — Run test 

In [19]:
f_stat, p_value_anova = stats.f_oneway(*groups)
print("F:", f_stat)
print("p-value:", p_value_anova)

F: 0.9151410065439705
p-value: 0.4698663379101742


#### **Output Interpretation**

The p-value is **greater than 0.05**. Therefore, we **fail to reject the null hypothesis ($H_0$)**.

This means there is **insufficient statistical evidence to conclude that the average customer tenure differs significantly across countries**.

> **Business Conclusion:** Based on the ANOVA test, no statistically significant difference in average customer tenure was detected across the countries in this dataset.

**Note:** Failing to reject the null hypothesis does not prove that the average tenure is exactly the same across all countries. It means that the observed differences are not statistically significant at the 0.05 significance level.

## **Test 4: Effect Size Analysis**

So far, the results of our statistical tests are:

- **Welch's t-test:** No statistically significant difference in mean `Balance` between churned and retained customers.
- **Chi-Square Test:** There is a statistically significant association between customer activity (`IsActive`) and churn (`Churned`).
- **ANOVA:** No statistically significant difference in mean `Tenure` across different countries.

Statistical significance tells us whether there is sufficient evidence of a relationship or difference, but it does not tell us **how large that difference or association is**.

Therefore, we use **Effect Size** to measure the magnitude of the observed differences or associations.

### **Cohen's d — Welch's t-test**

For the Welch's t-test, we compared the mean `Balance` between churned and retained customers.

**Cohen's d** can be used to quantify the standardized difference between the two group means.

The following guidelines can be used for interpretation:

| **Absolute Cohen's d** | **Effect Size** |
| ---------------------- | --------------- |
| < 0.2                  | Very Small      |
| 0.2 – < 0.5            | Small           |
| 0.5 – < 0.8            | Medium          |
| 0.8 – < 1.2            | Large           |
| ≥ 1.2                  | Very Large      |

The absolute value of Cohen's d is used to describe the **magnitude** of the effect, while the sign indicates the **direction** of the difference.

In [20]:
# means
churned_t_test_mean = churned_t_test.mean()
retained_t_test_mean = retained_t_test.mean()

# stds
churned_t_test_std = churned_t_test.std()
retained_t_test_std = retained_t_test.std()

# sample sizes
churned_t_test_n = len(churned_t_test)
retained_t_test_n = len(retained_t_test)

# pooled std
pooled_std = np.sqrt(
    ((churned_t_test_n-1)*churned_t_test_std**2 + (retained_t_test_n-1)*retained_t_test_std**2) / (churned_t_test_n+retained_t_test_n-2)
)

# cohen d
cohen_d = (churned_t_test_mean - retained_t_test_mean) / pooled_std

print("Cohen’s d:", cohen_d)

Cohen’s d: 0.03623904081079434


#### **Cohen's d — Output Interpretation**

The Welch's t-test indicated **no statistically significant difference** in average account balance between churned and retained customers (`p-value > 0.05`).

Additionally, the effect size (**Cohen's d = 0.03**) is very small/negligible. This indicates that the difference in average balance between the two customer groups is also **practically very small**.

| **Metric** | **Interpretation** |
|---|---|
| **High p-value (Welch's t-test)** | There is insufficient statistical evidence to conclude that the average balance differs between churned and retained customers. |
| **Cohen's d = 0.03** | The difference in average balance between the groups is negligible from a practical perspective. |

> 

### **Cramér's V — Chi-Square Test**

The Chi-Square test showed a statistically significant association between customer activity (`IsActive`) and customer churn (`Churned`).

However, the Chi-Square test only tells us whether an association exists. It does not tell us **how strong the association is**.

Therefore, **Cramér's V** is used to measure the strength of the association between the two categorical variables.

The following guidelines are used for interpretation:

| **Cramér's V** | **Effect Size** |
|---|---|
| < 0.1 | Negligible |
| 0.1 – < 0.3 | Weak |
| 0.3 – < 0.5 | Moderate |
| ≥ 0.5 | Strong |

n_chi = ct.sum().sum()
k_chi = min(ct.shape)

cramers_v = np.sqrt(chi2 / (n_chi * (k_chi-1)))

print("Cramér’s V:", cramers_v)

#### **Cramér's V — Output Interpretation**

The Chi-Square test shows a **statistically significant association** between customer activity (`IsActive`) and customer churn (`Churned`).

However, **Cramér's V = 0.15** indicates a **weak association**. This means that although customer activity and churn are statistically associated, the strength of this relationship is relatively small.

| **Metric** | **Interpretation** |
|---|---|
| **Small p-value (Chi-Square Test)** | There is a statistically significant association between customer activity and churn. |
| **Cramér's V = 0.15** | The association between customer activity and churn is weak. |

> **Business Conclusion:** Customer activity is associated with churn, but the relationship is weak. Therefore, activity status alone may not be sufficient to explain or identify customer churn, and other customer characteristics should also be considered.

**Note:** The Chi-Square test and Cramér's V measure association, not causation. Therefore, we cannot conclude that inactivity directly causes customer churn.

### **Eta Squared (η²) / Omega Squared (ω²) — ANOVA**

The ANOVA test determines whether there is a statistically significant difference in average `Tenure` across countries.

However, the ANOVA p-value does not tell us **how large the difference is**.

Therefore, **Eta Squared (η²)** or **Omega Squared (ω²)** can be used to measure the effect size and determine the practical magnitude of the differences in `Tenure` across countries.

The following guidelines can be used for interpretation:

| **η² / ω²** | **Effect Size** |
|---|---|
| < 0.01 | Negligible |
| 0.01 – < 0.06 | Small |
| 0.06 – < 0.14 | Medium |
| ≥ 0.14 | Large |

In [22]:
# Calculate SS
all_values = df_anova["Tenure"].values
grand_mean = np.mean(all_values)

ss_between = sum(
    len(group) * (np.mean(group) - grand_mean)**2
    for group in groups
)

ss_within = sum(
    sum((group - np.mean(group))**2)
    for group in groups
)

ss_total = ss_between + ss_within
print(ss_total)

83638.36159999996


In [23]:
# Calculate Eta score
k = len(groups)
n = len(all_values)

df_between = k - 1
df_within = n - k

ms_within = ss_within / df_within

eta_squared = ss_between / ss_total
print("Eta Squared (η²):", eta_squared)

Eta Squared (η²): 0.00045763568409215794


In [24]:
# Calculate Omega score
omega_squared = (
    ss_between - (df_between * ms_within)
) / (ss_total + ms_within)

print("Omega Squared (ω²):", omega_squared)

Omega Squared (ω²): -4.2431297066592226e-05


#### **Eta Squared / Omega Squared — Output Interpretation**

The One-Way ANOVA test indicates **no statistically significant difference** in average customer tenure across countries (`p-value > 0.05`).

The effect-size measures also indicate a negligible effect:

- **Eta Squared (η²) = 0.00046** → Negligible effect
- **Omega Squared (ω²) ≈ 0** → Negligible effect

The calculated Omega Squared value was slightly negative (`-0.000042`). Since a negative effect size does not represent a meaningful negative effect, it is interpreted as approximately **0**.

| **Metric** | **Interpretation** |
|---|---|
| **High p-value (ANOVA)** | There is insufficient statistical evidence to conclude that average tenure differs across countries. |
| **η² = 0.00046** | Country explains a negligible proportion of the variation in customer tenure. |
| **ω² ≈ 0** | The estimated practical effect of country on customer tenure is negligible. |

> **Business Conclusion:** Both the ANOVA result and the effect-size measures indicate that geographic region has little to no meaningful relationship with customer tenure in this dataset. Therefore, `Geography` alone may not be useful for distinguishing customers based on tenure.